# Imports

In [2]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
import re
import json

# Load Dataset

In [3]:
dataset = load_dataset(
    "zefang-liu/phishing-email-dataset"
)

dataset

DatasetDict({
    train: Dataset({
        features: ['Unnamed: 0', 'Email Text', 'Email Type'],
        num_rows: 18650
    })
})

# Convert Dataset To Pandas DataFrame

In [6]:
malicious_email_df = pd.DataFrame(
    dataset["train"]
)

malicious_email_df.head(20)


,Unnamed: 0,Email Text,Email Type
0,0,"re : 6 . 1100 , disc : uniformitarianism , re ...",Safe Email
1,1,the other side of * galicismos * * galicismo *...,Safe Email
2,2,re : equistar deal tickets are you still avail...,Safe Email
3,3,\nHello I am your hot lil horny toy.\n I am...,Phishing Email
4,4,software at incredibly low prices ( 86 % lower...,Phishing Email
5,5,global risk management operations sally congra...,Safe Email
6,6,"On Sun, Aug 11, 2002 at 11:17:47AM +0100, wint...",Safe Email
7,7,"entourage , stockmogul newsletter ralph velez ...",Phishing Email
8,8,"we owe you lots of money dear applicant , afte...",Phishing Email
9,9,re : coastal deal - with exxon participation u...,Safe Email


In [7]:
malicious_email_df = malicious_email_df.drop(
    columns=["Unnamed: 0"]
)

# Dataset Shape

In [8]:
print("Dataset shape:")
print(malicious_email_df.shape)

Dataset shape:
(18650, 2)


# Null values

In [19]:
print(f"Null Email Type Values: {malicious_email_df["Email Type"].isnull().sum()}")

Null Email Type Values: 0


In [20]:
print(f"Null Email Text Values: {malicious_email_df["Email Text"].isnull().sum()}")

Null Email Text Values: 16


# Delete Null Values

In [26]:
malicious_email_df = malicious_email_df.dropna()
print("Dataset shape:")
print(malicious_email_df.shape)


Dataset shape:
(18634, 2)


# Display Email Type Distribution

In [23]:
malicious_email_df["Email Type"].value_counts()

Email Type
Safe Email        11322
Phishing Email     7328
Name: count, dtype: int64

# Display A Phishing Email Example

In [12]:
malicious_email_example = malicious_email_df[
    malicious_email_df["Email Type"] == "Phishing Email"
].iloc[4]

print(malicious_email_example["Email Text"])

make her beg you to give it to her everynight you too can please you and your partner so much better . and it doesn ' t take half a year like other solutions . if you don ' t trust me , trust the new york times , newsweek , time magazine , and the testemonials of many people all over the net . look , if anything else check out the site and see the facts for yourself its new , its safe ! its the most advanced penile enlargement solution ! it ' s 100 % guaranteed to enlarge your penis . 3 + inches magna - rx patch - click here the amazing , new magna rx patch is not available in any stores or on other websites . accept no imitations ! order your male enhancement patch now through this exclusive website offer and get a 1 - month supply free ! one small investment in yourself will last a lifetime ! - no pills or capsules - no lotions or cremes - no pumps , weights , or exercises - no prescription necessary - doctor designed endorsed - 100 % safe natural - more stamina energy - increase pen

# Define Text Cleaning Function

In [13]:
STOP_WORDS = set(
    ENGLISH_STOP_WORDS
)

In [14]:
def clean_text(text: str) -> str:
    """
    Cleans email text for malicious email EDA analysis.
    """

    text = text.lower()

    text = re.sub(r"http\S+", " ", text)

    text = re.sub(r"\bre\b", " ", text)

    text = re.sub(r"\bfw\b", " ", text)

    text = re.sub(r"\b\d+\b", " ", text)

    text = re.sub(r"[^a-z\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    words = text.split()

    filtered_words = [
        word
        for word in words
        if word not in STOP_WORDS
        and len(word) > 2
    ]

    return " ".join(filtered_words)

# Split Malicious and Legitimate emails

In [27]:
malicious_only_df = malicious_email_df[
    malicious_email_df["Email Type"] == "Phishing Email"
]

legitimate_only_df = malicious_email_df[
    malicious_email_df["Email Type"] != "Phishing Email"
]

# Combine Email Text

In [28]:
all_malicious_text = " ".join(
    malicious_only_df["Email Text"].astype(str)
)

all_legitimate_text = " ".join(
    legitimate_only_df["Email Text"].astype(str)
)

# Clean Malicious Email Text

In [29]:
cleaned_malicious_text = clean_text(
    all_malicious_text
)

cleaned_legitimate_text = clean_text(
    all_legitimate_text
)

# Split Text Into Words

In [30]:
malicious_words = cleaned_malicious_text.split()
legitimate_words = cleaned_legitimate_text.split()

# Count Word Frequencies

In [31]:
malicious_counter = Counter(
    malicious_words
)

legitimate_counter = Counter(
    legitimate_words
)

# Display Top 50 Words in Malicious Emails

In [38]:
top_malicious_words = malicious_counter.most_common(50)

top_malicious_words_df = pd.DataFrame(
    top_malicious_words,
    columns=["word", "count"]
)

top_malicious_words_df

,word,count
0,email,5397
1,com,4899
2,free,4530
3,mail,3971
4,money,3929
5,company,3915
6,information,3895
7,business,3622
8,time,3384
9,report,3160


# Calculate Total Word Counts

In [40]:
total_malicious_words = len(
    malicious_words
)

total_legitimate_words = len(
    legitimate_words
)

print("Total words in malicious emails:", total_malicious_words)
print("Total words in legitimate emails:", total_legitimate_words)

Total words in malicious emails: 928572
Total words in legitimate emails: 3067780


# Normalize Word Counts by number of Words

In [41]:
malicious_frequencies = {
    word: count / total_malicious_words
    for word, count in malicious_counter.items()
}

legitimate_frequencies = {
    word: count / total_legitimate_words
    for word, count in legitimate_counter.items()
}

# Normalize Word Counts by number of Emails

In [48]:
num_malicious_emails = len(malicious_only_df)
num_legitimate_emails = len(legitimate_only_df)

malicious_frequencies_per_email = {
    word: count / num_malicious_emails
    for word, count in malicious_counter.items()
}

legitimate_frequencies_per_email = {
    word: count / num_legitimate_emails
    for word, count in legitimate_counter.items()
}

# Build Combined Vocabulary

In [42]:
all_words = set(malicious_frequencies.keys()).union(legitimate_frequencies.keys())

# Top Words by Normalize with number of words

In [46]:
word_scores = {
    word:
        malicious_frequencies.get(word, 0)
        -
        legitimate_frequencies.get(word, 0)

    for word in all_words
}

top_suspicious_words = sorted(
word_scores.items(),
key=lambda item: item[1],
reverse=True)

top_suspicious_words[:50]

[('email', 0.0036998736394399253),
 ('free', 0.0036032693115517706),
 ('money', 0.0033556764635159094),
 ('click', 0.0029867805263770167),
 ('business', 0.002600650144660358),
 ('company', 0.002595109325515873),
 ('com', 0.0022948599162159653),
 ('report', 0.0022436044626838857),
 ('statements', 0.0020782612882884665),
 ('order', 0.0014967738752156114),
 ('receive', 0.001491168229679278),
 ('http', 0.0014409505937855514),
 ('make', 0.0014116259688548695),
 ('online', 0.0013975155730713163),
 ('internet', 0.00138133944984611),
 ('save', 0.0013281483709909978),
 ('time', 0.0013048286944801735),
 ('investment', 0.0013008984040335646),
 ('home', 0.0012824335779320086),
 ('offer', 0.0012628169758347606),
 ('price', 0.0012272817067856589),
 ('just', 0.0012252454795858739),
 ('best', 0.0012226656455920338),
 ('future', 0.0011851129090761838),
 ('software', 0.0011834661162416756),
 ('remove', 0.0011767885948839947),
 ('send', 0.001118604993611712),
 ('font', 0.0010896892760985334),
 ('want', 0

# Top Words by Normalize with number of emails

In [49]:
word_scores_by_email_count = {
    word:
        malicious_frequencies_per_email.get(word, 0)
        -
        legitimate_frequencies_per_email.get(word, 0)
    for word in all_words
}

top_suspicious_words_by_email_count = sorted(
    word_scores_by_email_count.items(),
    key=lambda item: item[1],
    reverse=True
)

top_suspicious_words_by_email_count[:50]

[('click', 0.32758531636282956),
 ('money', 0.30009864897720484),
 ('free', 0.2740075479005843),
 ('statements', 0.25195778382321055),
 ('email', 0.16576470762176776),
 ('business', 0.1431155218804852),
 ('save', 0.13863152797539463),
 ('adobe', 0.12931377284093207),
 ('online', 0.12843486104201768),
 ('investment', 0.12287586047883382),
 ('report', 0.11799919368461009),
 ('remove', 0.11157294989673673),
 ('receive', 0.10822938397272289),
 ('offer', 0.10123959395101113),
 ('company', 0.09618787438487525),
 ('securities', 0.09532082442849595),
 ('font', 0.09215878576478395),
 ('pills', 0.09201769989837952),
 ('removed', 0.08851099136206614),
 ('dollars', 0.08478444012296504),
 ('act', 0.08364758760562596),
 ('future', 0.08344091130646672),
 ('viagra', 0.08328206891406789),
 ('website', 0.08150532918038389),
 ('advice', 0.07719650883989923),
 ('products', 0.07513208922656729),
 ('product', 0.07450821912142547),
 ('professional', 0.07132762669993974),
 ('low', 0.07083778816788211),
 ('siz

# Convert to DataFrames

In [50]:
top_suspicious_words_df = pd.DataFrame(
    top_suspicious_words[:50],
    columns=["word", "score"]
)

top_suspicious_words_df

,word,score
0,email,0.003700
1,free,0.003603
2,money,0.003356
3,click,0.002987
4,business,0.002601
5,company,0.002595
6,com,0.002295
7,report,0.002244
8,statements,0.002078
9,order,0.001497


In [51]:
top_suspicious_words_by_email_count_df = pd.DataFrame(
    top_suspicious_words_by_email_count[:50],
    columns=["word", "score_per_email"]
)

top_suspicious_words_by_email_count_df

,word,score_per_email
0,click,0.327585
1,money,0.300099
2,free,0.274008
3,statements,0.251958
4,email,0.165765
5,business,0.143116
6,save,0.138632
7,adobe,0.129314
8,online,0.128435
9,investment,0.122876
